# Qwen3 Proxy Label Fine-Tuning in Colab

This notebook fine-tunes an LLM on your candidate dataset using these labels:

- `0-30 -> low_match`
- `31-70 -> medium_match`
- `71-100 -> high_match`

Use this notebook with the Excel file `Potential_Talents_Proxy_Labels.xlsx`.

Recommended runtime:
- GPU runtime in Colab
- `Qwen/Qwen3-8B` if you have enough VRAM
- if memory is too small, change the model to a smaller one before training

In [1]:
import subprocess
import sys

# Colab sometimes ships an older optional torchao build that breaks PEFT adapter loading.
# This notebook uses bitsandbytes QLoRA, so removing torchao is the simplest compatible fix.
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'], check=False)
subprocess.run([
    sys.executable,
    '-m',
    'pip',
    'install',
    '-q',
    'transformers>=4.57.0',
    'trl>=0.24.0',
    'peft>=0.17.0',
    'accelerate>=1.10.0',
    'datasets>=4.0.0',
    'bitsandbytes>=0.47.0',
    'openpyxl>=3.1.0',
], check=True)

print('Dependencies installed. If this runtime had already imported torchao, restart the runtime once and rerun from the top.')

Dependencies installed. If this runtime had already imported torchao, restart the runtime once and rerun from the top.


## 1. Upload your Excel file

When you run the next cell, upload:

- `Potential_Talents_Proxy_Labels.xlsx`

In [2]:
from google.colab import files

uploaded = files.upload()
INPUT_XLSX = list(uploaded.keys())[0]
print('Uploaded file:', INPUT_XLSX)

MODEL_NAME = 'Qwen/Qwen3-0.6B'
OUTPUT_DIR = '/content/qwen3_proxy_labels'
DATA_DIR = '/content/proxy_label_data'

SEED = 42
NUM_EPOCHS = 3
LEARNING_RATE = 2e-4
TRAIN_BATCH_SIZE = 1
EVAL_BATCH_SIZE = 1
GRAD_ACCUMULATION = 4
MAX_LENGTH = 64

print('Model:', MODEL_NAME)

ModuleNotFoundError: No module named 'google.colab'

## 2. Build the training dataset

This converts the Excel sheet into the exact Hugging Face TRL format:

- `prompt`: system + user
- `completion`: assistant label

In [ ]:
import json
import os
import random
from collections import Counter, defaultdict

from openpyxl import load_workbook

SYSTEM_MESSAGE = (
    'You are a recruiting screening model. '
    'Read the candidate information and return only one label: '
    'low_match, medium_match, or high_match.'
)


def normalize_text(value):
    if value is None:
        return ''
    return ' '.join(str(value).strip().split())


def load_rows(input_xlsx, sheet_name='Sheet1'):
    workbook = load_workbook(input_xlsx, data_only=True)
    sheet = workbook[sheet_name]
    rows = list(sheet.iter_rows(values_only=True))
    headers = [normalize_text(cell) for cell in rows[0]]
    header_index = {name: idx for idx, name in enumerate(headers)}

    required = {'id', 'title', 'location', 'screening_score', 'match_label'}
    missing = sorted(required - set(header_index))
    if missing:
        raise ValueError(f'Missing required columns: {missing}')

    records = []
    for raw_row in rows[1:]:
        title = normalize_text(raw_row[header_index['title']])
        location = normalize_text(raw_row[header_index['location']])
        match_label = normalize_text(raw_row[header_index['match_label']])

        if not title or not match_label:
            continue

        records.append({
            'id': raw_row[header_index['id']],
            'title': title,
            'location': location,
            'screening_score': raw_row[header_index['screening_score']],
            'match_label': match_label,
        })
    return records


def build_example(row):
    user_message = (
        'Classify the candidate into exactly one label: '
        'low_match, medium_match, or high_match.\n\n'
        f"Candidate title: {row['title']}\n"
        f"Candidate location: {row['location']}"
    )
    return {
        'prompt': [
            {'role': 'system', 'content': SYSTEM_MESSAGE},
            {'role': 'user', 'content': user_message},
        ],
        'completion': [
            {'role': 'assistant', 'content': row['match_label']},
        ],
        'metadata': {
            'id': row['id'],
            'screening_score': row['screening_score'],
        },
    }


def stratified_split(rows, train_ratio=0.8, validation_ratio=0.1, test_ratio=0.1, seed=42):
    grouped = defaultdict(list)
    for row in rows:
        grouped[row['match_label']].append(row)

    rng = random.Random(seed)
    split_rows = {'train': [], 'validation': [], 'test': []}

    for label, items in grouped.items():
        rng.shuffle(items)
        n_items = len(items)
        n_train = int(n_items * train_ratio)
        n_validation = int(n_items * validation_ratio)
        n_test = n_items - n_train - n_validation

        if n_items >= 3:
            if n_train == 0:
                n_train = 1
            if n_validation == 0:
                n_validation = 1
                n_train = max(1, n_train - 1)
            if n_test == 0:
                n_test = 1
                if n_train > 1:
                    n_train -= 1
                else:
                    n_validation = max(1, n_validation - 1)

        split_rows['train'].extend(items[:n_train])
        split_rows['validation'].extend(items[n_train:n_train + n_validation])
        split_rows['test'].extend(items[n_train + n_validation:n_train + n_validation + n_test])

    return split_rows


def write_jsonl(path, rows):
    with open(path, 'w', encoding='utf-8') as handle:
        for row in rows:
            handle.write(json.dumps(build_example(row), ensure_ascii=False) + '\n')


rows = load_rows(INPUT_XLSX)
splits = stratified_split(rows, seed=SEED)

os.makedirs(DATA_DIR, exist_ok=True)
for split_name, split_rows in splits.items():
    write_jsonl(os.path.join(DATA_DIR, f'{split_name}.jsonl'), split_rows)

summary = {
    split_name: {
        'count': len(split_rows),
        'label_counts': dict(Counter(row['match_label'] for row in split_rows)),
    }
    for split_name, split_rows in splits.items()
}

print(json.dumps(summary, indent=2))
print('\nExample training row:')
print(json.dumps(build_example(rows[0]), indent=2))

## 3. Fine-tune Qwen3-8B with QLoRA

This uses:
- `BitsAndBytesConfig` for 4-bit loading
- `LoRA` adapters
- `SFTTrainer` from TRL

Important:
- If Colab runs out of memory, lower `MAX_LENGTH`
- or switch `MODEL_NAME` to a smaller model before running this cell

In [ ]:
import torch
from datasets import load_dataset
from peft import LoraConfig
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from trl import SFTConfig, SFTTrainer

if not torch.cuda.is_available():
    raise RuntimeError('Please switch Colab to a GPU runtime first.')

compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

dataset = load_dataset(
    'json',
    data_files={
        'train': os.path.join(DATA_DIR, 'train.jsonl'),
        'validation': os.path.join(DATA_DIR, 'validation.jsonl'),
    },
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quantization_config,
    device_map='auto',
    torch_dtype=compute_dtype,
    trust_remote_code=True,
)
model.config.use_cache = False

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=[
        'q_proj',
        'k_proj',
        'v_proj',
        'o_proj',
        'gate_proj',
        'up_proj',
        'down_proj',
    ],
)

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUMULATION,
    max_length=MAX_LENGTH,
    logging_steps=10,
    save_strategy='epoch',
    eval_strategy='epoch',
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    report_to='none',
    completion_only_loss=True,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset['train'],
    eval_dataset=dataset['validation'],
    processing_class=tokenizer,
    peft_config=peft_config,
)

trainer.train()
trainer.save_model(os.path.join(OUTPUT_DIR, 'final_adapter'))
tokenizer.save_pretrained(os.path.join(OUTPUT_DIR, 'final_adapter'))

print('Training finished.')
print('Saved adapter to:', os.path.join(OUTPUT_DIR, 'final_adapter'))

## 4. Test the fine-tuned adapter

If you previously hit a `torchao` compatibility error, run the first setup cell again and then rerun the notebook from the top before testing inference.

In [ ]:
from peft import AutoPeftModelForCausalLM

adapter_path = os.path.join(OUTPUT_DIR, 'final_adapter')

inference_tokenizer = AutoTokenizer.from_pretrained(adapter_path, trust_remote_code=True)
inference_model = AutoPeftModelForCausalLM.from_pretrained(
    adapter_path,
    device_map='auto',
    torch_dtype=compute_dtype,
    trust_remote_code=True,
)
inference_model.eval()

messages = [
    {
        'role': 'system',
        'content': 'You are a recruiting screening model. Read the candidate information and return only one label: low_match, medium_match, or high_match.'
    },
    {
        'role': 'user',
        'content': 'Classify the candidate into exactly one label: low_match, medium_match, or high_match.\n\nCandidate title: Data analyst with Python, SQL, Power BI, Tableau, and machine learning experience\nCandidate location: United States'
    },
]

text = inference_tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)
inputs = inference_tokenizer(text, return_tensors='pt').to(inference_model.device)

with torch.no_grad():
    outputs = inference_model.generate(**inputs, max_new_tokens=8)

generated = inference_tokenizer.decode(outputs[0][inputs['input_ids'].shape[-1]:], skip_special_tokens=True)
print('Model prediction:', generated.strip())